# MultinomialNB phân loại topic cho `Clause`

Notebook này:
- Đọc file `sample_set_Labeled.csv`
- Dùng `x = Clause`, `y = code`
- Chia dữ liệu `80% train - 10% valid - 10% test`
- Tuning hyperparameter cho `TfidfVectorizer + MultinomialNB`
- Chọn mô hình theo **validation macro-F1 có phạt overfit**
- Đánh giá bằng `accuracy, precision, recall, f1-score` cho từng lớp và toàn mô hình ở `train`, `valid`, `test`
- Lưu mô hình tốt nhất ra file `.pkl`


In [1]:
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    precision_recall_fscore_support,
)
from sklearn.model_selection import ParameterGrid, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

In [2]:
# =========================
# 1) Cấu hình
# =========================
RANDOM_STATE = 42
DATA_PATH = r"/content/annotated_clauses_topics.csv"   # đổi nếu file ở chỗ khác
TEXT_COL = "Clause"
TARGET_COL = "code"

MODEL_PATH = "best_multinomial_nb_topic_classifier.pkl"
METRICS_JSON_PATH = "best_multinomial_nb_metrics.json"
TUNING_CSV_PATH = "tuning_results.csv"

In [3]:
# =========================
# 2) Hàm hỗ trợ
# =========================
def print_header(title: str):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)

def evaluate_model(model, X, y, label_order, stage_name: str):
    y_pred = model.predict(X)

    accuracy = accuracy_score(y, y_pred)
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y, y_pred, average="macro", zero_division=0
    )
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(
        y, y_pred, average="weighted", zero_division=0
    )

    report = classification_report(
        y,
        y_pred,
        labels=label_order,
        output_dict=True,
        zero_division=0,
    )

    result = {
        "stage": stage_name,
        "accuracy": float(accuracy),
        "macro_precision": float(macro_p),
        "macro_recall": float(macro_r),
        "macro_f1": float(macro_f1),
        "weighted_precision": float(weighted_p),
        "weighted_recall": float(weighted_r),
        "weighted_f1": float(weighted_f1),
        "classification_report": report,
    }
    return result

def print_metrics(result, label_order):
    print_header(f"METRICS - {result['stage']}")
    print(f"Accuracy           : {result['accuracy']:.4f}")
    print(f"Macro Precision    : {result['macro_precision']:.4f}")
    print(f"Macro Recall       : {result['macro_recall']:.4f}")
    print(f"Macro F1           : {result['macro_f1']:.4f}")
    print(f"Weighted Precision : {result['weighted_precision']:.4f}")
    print(f"Weighted Recall    : {result['weighted_recall']:.4f}")
    print(f"Weighted F1        : {result['weighted_f1']:.4f}")

    print("\nPer-class metrics:")
    print(f"{'Class':<10}{'Precision':>12}{'Recall':>12}{'F1-score':>12}{'Support':>12}")
    print("-" * 58)
    for label in label_order:
        row = result["classification_report"].get(label, {})
        print(
            f"{label:<10}"
            f"{row.get('precision', 0):>12.4f}"
            f"{row.get('recall', 0):>12.4f}"
            f"{row.get('f1-score', 0):>12.4f}"
            f"{int(row.get('support', 0)):>12}"
        )

def compute_overfit_gap(train_result, valid_result):
    return train_result["macro_f1"] - valid_result["macro_f1"]

def build_pipeline(params):
    vectorizer = TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=params["tfidf__ngram_range"],
        min_df=params["tfidf__min_df"],
        max_df=params["tfidf__max_df"],
        sublinear_tf=params["tfidf__sublinear_tf"],
        use_idf=params["tfidf__use_idf"],
        norm=params["tfidf__norm"],
    )

    nb = MultinomialNB(
        alpha=params["nb__alpha"],
        fit_prior=params["nb__fit_prior"],
    )

    pipe = Pipeline([
        ("tfidf", vectorizer),
        ("nb", nb),
    ])
    return pipe

In [4]:
# =========================
# 3) Đọc và làm sạch dữ liệu
# =========================
data_file = Path(DATA_PATH)
if not data_file.exists():
    raise FileNotFoundError(f"Không tìm thấy file dữ liệu: {data_file.resolve()}")

df = pd.read_csv(data_file)

required_cols = {TEXT_COL, TARGET_COL}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Thiếu cột bắt buộc: {missing}")

df = df[[TEXT_COL, TARGET_COL]].copy()
df = df.dropna(subset=[TEXT_COL, TARGET_COL])
df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
df[TARGET_COL] = df[TARGET_COL].astype(str).str.strip()
df = df[(df[TEXT_COL] != "") & (df[TARGET_COL] != "")].reset_index(drop=True)

print_header("LOAD DATA")
print(f"Shape after cleaning: {df.shape}")
print("\n5 dòng đầu:")
display(df.head())

print("\nPhân bố lớp:")
display(df[TARGET_COL].value_counts().sort_index().to_frame("count"))


LOAD DATA
Shape after cleaning: (23173, 2)

5 dòng đầu:


,Clause,code
0,The lake is as green as paint.,EL
1,but the 12km hike (there and back) was well wo...,EL
2,It was quite crowded,TCr
3,but they actually physically blocked us from L...,SE
4,Example rice sheets for roll in the market 800...,TC



Phân bố lớp:


,count
code,
CA,898
CT,1475
EC,460
EL,698
HL,1477
LC,1734
RA,1697
SE,1411
TC,1872


In [5]:
# =========================
# 4) Chia train / valid / test = 80 / 10 / 10
# =========================
X = df[TEXT_COL]
y = df[TARGET_COL]
label_order = sorted(y.unique())

# Bước 1: train 80%, temp 20%
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

# Bước 2: temp 20% -> valid 10%, test 10%
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=RANDOM_STATE,
)

print_header("DATA SPLIT")
print(f"Train size: {len(X_train)} ({len(X_train)/len(df):.2%})")
print(f"Valid size: {len(X_valid)} ({len(X_valid)/len(df):.2%})")
print(f"Test  size: {len(X_test)} ({len(X_test)/len(df):.2%})")

split_df = pd.DataFrame({
    "subset": ["train", "valid", "test"],
    "size": [len(X_train), len(X_valid), len(X_test)],
    "ratio": [len(X_train)/len(df), len(X_valid)/len(df), len(X_test)/len(df)],
})
display(split_df)


DATA SPLIT
Train size: 18538 (80.00%)
Valid size: 2317 (10.00%)
Test  size: 2318 (10.00%)


,subset,size,ratio
0,train,18538,0.799983
1,valid,2317,0.099987
2,test,2318,0.100030


## Tuning hyperparameter

Ở đây mình không chọn tham số cố định ngay từ đầu.

Tiêu chí chọn mô hình:
- ưu tiên **validation macro-F1**
- nhưng có **phạt overfit** nếu `train_macro_f1 - valid_macro_f1` quá lớn

Công thức:
- `selection_score = valid_macro_f1 - penalty`
- `penalty = GAP_PENALTY * max(0, gap - GAP_THRESHOLD)`

Như vậy mô hình sẽ bớt bị chọn theo kiểu điểm train rất cao nhưng validation không đủ tốt.


In [6]:
# =========================
# 5) Hyperparameter tuning
# =========================
param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [2, 3, 5],
    "tfidf__max_df": [0.90, 0.95, 1.0],
    "tfidf__sublinear_tf": [False, True],
    "tfidf__use_idf": [True],
    "tfidf__norm": ["l2"],
    "nb__alpha": [0.05, 0.1, 0.3, 0.5, 1.0, 2.0, 5.0],
    "nb__fit_prior": [True, False],
}

grid = list(ParameterGrid(param_grid))
print_header("HYPERPARAMETER TUNING")
print(f"Number of candidate models: {len(grid)}")

tuning_rows = []
best_model = None
best_params = None
best_score = -np.inf
best_train_result = None
best_valid_result = None

# Chống overfit
GAP_THRESHOLD = 0.03
GAP_PENALTY = 0.50

for idx, params in enumerate(grid, start=1):
    model = build_pipeline(params)
    model.fit(X_train, y_train)

    train_result = evaluate_model(model, X_train, y_train, label_order, "TRAIN")
    valid_result = evaluate_model(model, X_valid, y_valid, label_order, "VALID")

    gap = compute_overfit_gap(train_result, valid_result)
    penalty = GAP_PENALTY * max(0.0, gap - GAP_THRESHOLD)
    selection_score = valid_result["macro_f1"] - penalty

    tuning_rows.append({
        "candidate": idx,
        "valid_macro_f1": valid_result["macro_f1"],
        "train_macro_f1": train_result["macro_f1"],
        "overfit_gap": gap,
        "selection_score": selection_score,
        **params,
    })

    print(
        f"[{idx:03d}/{len(grid)}] "
        f"valid_macro_f1={valid_result['macro_f1']:.4f} | "
        f"train_macro_f1={train_result['macro_f1']:.4f} | "
        f"gap={gap:.4f} | score={selection_score:.4f} | params={params}"
    )

    if selection_score > best_score:
        best_score = selection_score
        best_model = model
        best_params = params
        best_train_result = train_result
        best_valid_result = valid_result

tuning_df = pd.DataFrame(tuning_rows).sort_values(
    by=["selection_score", "valid_macro_f1"], ascending=False
).reset_index(drop=True)

print_header("BEST PARAMS")
print(json.dumps(best_params, indent=2, default=str))
print(f"Best selection score: {best_score:.4f}")

display(tuning_df.head(10))


HYPERPARAMETER TUNING
Number of candidate models: 504
[001/504] valid_macro_f1=0.6017 | train_macro_f1=0.8824 | gap=0.2806 | score=0.4764 | params={'nb__alpha': 0.05, 'nb__fit_prior': True, 'tfidf__max_df': 0.9, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 1), 'tfidf__norm': 'l2', 'tfidf__sublinear_tf': False, 'tfidf__use_idf': True}
[002/504] valid_macro_f1=0.6038 | train_macro_f1=0.8862 | gap=0.2824 | score=0.4776 | params={'nb__alpha': 0.05, 'nb__fit_prior': True, 'tfidf__max_df': 0.9, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 1), 'tfidf__norm': 'l2', 'tfidf__sublinear_tf': True, 'tfidf__use_idf': True}
[003/504] valid_macro_f1=0.6007 | train_macro_f1=0.9700 | gap=0.3694 | score=0.4310 | params={'nb__alpha': 0.05, 'nb__fit_prior': True, 'tfidf__max_df': 0.9, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 2), 'tfidf__norm': 'l2', 'tfidf__sublinear_tf': False, 'tfidf__use_idf': True}
[004/504] valid_macro_f1=0.5955 | train_macro_f1=0.9707 | gap=0.3753 | score=0.4228 | params={'nb

,candidate,valid_macro_f1,train_macro_f1,overfit_gap,selection_score,nb__alpha,nb__fit_prior,tfidf__max_df,tfidf__min_df,tfidf__ngram_range,tfidf__norm,tfidf__sublinear_tf,tfidf__use_idf
0,333,0.663603,0.761837,0.098234,0.629487,1.0,False,0.90,5,"(1, 1)",l2,False,True
1,345,0.663603,0.761837,0.098234,0.629487,1.0,False,0.95,5,"(1, 1)",l2,False,True
2,357,0.663603,0.761837,0.098234,0.629487,1.0,False,1.00,5,"(1, 1)",l2,False,True
3,334,0.663065,0.766322,0.103257,0.626437,1.0,False,0.90,5,"(1, 1)",l2,True,True
4,346,0.663065,0.766322,0.103257,0.626437,1.0,False,0.95,5,"(1, 1)",l2,True,True
5,358,0.663065,0.766322,0.103257,0.626437,1.0,False,1.00,5,"(1, 1)",l2,True,True
6,262,0.667438,0.786086,0.118649,0.623114,0.5,False,0.90,5,"(1, 1)",l2,True,True
7,274,0.667438,0.786086,0.118649,0.623114,0.5,False,0.95,5,"(1, 1)",l2,True,True
8,286,0.667438,0.786086,0.118649,0.623114,0.5,False,1.00,5,"(1, 1)",l2,True,True
9,261,0.665936,0.782008,0.116072,0.622900,0.5,False,0.90,5,"(1, 1)",l2,False,True


In [7]:
# =========================
# 6) Đánh giá mô hình tốt nhất trên test
# =========================
test_result = evaluate_model(best_model, X_test, y_test, label_order, "TEST")

print_metrics(best_train_result, label_order)
print_metrics(best_valid_result, label_order)
print_metrics(test_result, label_order)


METRICS - TRAIN
Accuracy           : 0.7965
Macro Precision    : 0.7769
Macro Recall       : 0.7645
Macro F1           : 0.7618
Weighted Precision : 0.8023
Weighted Recall    : 0.7965
Weighted F1        : 0.7952

Per-class metrics:
Class        Precision      Recall    F1-score     Support
----------------------------------------------------------
CA              0.7481      0.8106      0.7781         718
CT              0.7671      0.8958      0.8264        1180
EC              0.8233      0.5951      0.6909         368
EL              0.7980      0.7294      0.7622         558
HL              0.7149      0.8613      0.7813        1182
LC              0.7403      0.8818      0.8049        1387
RA              0.7750      0.8446      0.8083        1358
SE              0.8168      0.7857      0.8009        1129
TC              0.7542      0.8605      0.8039        1498
TCr             0.7847      0.7319      0.7574         787
UH              0.7415      0.7902      0.7651         882


In [8]:
# =========================
# 7) Kiểm tra overfitting
# =========================
print_header("OVERFITTING CHECK")
train_valid_gap = best_train_result["macro_f1"] - best_valid_result["macro_f1"]
valid_test_gap = best_valid_result["macro_f1"] - test_result["macro_f1"]

print(f"Train - Valid macro F1 gap: {train_valid_gap:.4f}")
print(f"Valid - Test  macro F1 gap: {valid_test_gap:.4f}")

if train_valid_gap <= 0.03:
    print("=> Dấu hiệu overfit thấp.")
elif train_valid_gap <= 0.06:
    print("=> Có chênh lệch nhẹ, cần theo dõi nhưng chưa quá nghiêm trọng.")
else:
    print("=> Có dấu hiệu overfit rõ hơn, nên cân nhắc tăng regularization / giảm độ phức tạp.")


OVERFITTING CHECK
Train - Valid macro F1 gap: 0.0982
Valid - Test  macro F1 gap: 0.0149
=> Có dấu hiệu overfit rõ hơn, nên cân nhắc tăng regularization / giảm độ phức tạp.


In [9]:
# =========================
# 8) Lưu model + metadata
# =========================
artifact = {
    "model_type": "MultinomialNB",
    "pipeline": best_model,
    "best_params": best_params,
    "text_column": TEXT_COL,
    "target_column": TARGET_COL,
    "label_order": label_order,
    "split": {
        "train": len(X_train),
        "valid": len(X_valid),
        "test": len(X_test),
        "random_state": RANDOM_STATE,
    },
}

joblib.dump(artifact, MODEL_PATH)

metrics_payload = {
    "best_params": {k: str(v) if isinstance(v, tuple) else v for k, v in best_params.items()},
    "train": best_train_result,
    "valid": best_valid_result,
    "test": test_result,
    "overfit": {
        "train_valid_macro_f1_gap": float(train_valid_gap),
        "valid_test_macro_f1_gap": float(valid_test_gap),
    },
}

with open(METRICS_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, ensure_ascii=False, indent=2)

tuning_df.to_csv(TUNING_CSV_PATH, index=False, encoding="utf-8-sig")

print_header("FILES SAVED")
print(f"Model file   : {Path(MODEL_PATH).resolve()}")
print(f"Metrics file : {Path(METRICS_JSON_PATH).resolve()}")
print(f"Tuning file  : {Path(TUNING_CSV_PATH).resolve()}")


FILES SAVED
Model file   : /content/best_multinomial_nb_topic_classifier.pkl
Metrics file : /content/best_multinomial_nb_metrics.json
Tuning file  : /content/tuning_results.csv


In [10]:
# =========================
# 9) Load model và dự đoán thử
# =========================
loaded = joblib.load(MODEL_PATH)

demo_texts = [
    "Only stayed for 15mins",
    "Take a scooter from Hoi An",
    "Great places to chill all day",
]

demo_preds = loaded["pipeline"].predict(demo_texts)

for text, pred in zip(demo_texts, demo_preds):
    print(f"Text: {text}")
    print(f"=> Predicted code: {pred}")
    print()

Text: Only stayed for 15mins
=> Predicted code: VE

Text: Take a scooter from Hoi An
=> Predicted code: VE

Text: Great places to chill all day
=> Predicted code: LC

